# RizeHire: BERT + ANN Integrated Model for Resume Classification
**Sandeep Kumar (2022UG1113) & Nikhil Sonkar (2022UG1111)**  
**IIIT Ranchi | Major Project | 2026**

This notebook trains an Artificial Neural Network (ANN) on top of BERT embeddings for resume-job description classification.

**Pipeline:** Raw Text → BERT Tokenization → Sentence-BERT Embeddings (384-dim) → Concatenation (768-dim) → ANN (768→512→256→128→2) → Classification

---

## Step 1: Setup & Install Dependencies

In [ ]:
!pip install -q sentence-transformers torch scikit-learn matplotlib seaborn pandas numpy kagglehub

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc)
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## Step 2: Load & Prepare Dataset
Using Kaggle recruitment dataset with 9,500+ resume-job pairs

In [ ]:
# Upload your dataset.csv from your RizeHire project folder
from google.colab import files
import pandas as pd

print("Upload your dataset.csv file from the RizeHire/python-explainability folder:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

print(f"\nLoaded {len(df)} rows")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Explore and prepare the data
print("Dataset Info:")
print(f"Total samples: {len(df)}")
print(f"\nColumn types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum())

# Display value counts for the target variable
# Adjust column names based on actual dataset
print(f"\nFirst few rows:")
print(df.head(3).to_string())

In [ ]:
# Dataset columns: ID, Name, Role, Transcript, Resume, decision, Reason_for_decision, Job_Description

resume_col = 'Resume'
job_col = 'Job_Description'
label_col = 'decision'

print(f"Resume column: {resume_col}")
print(f"Job column: {job_col}")
print(f"Label column: {label_col}")
print(f"\nLabel values: {df[label_col].value_counts().to_dict()}")
print(f"Sample resume (first 200 chars): {df[resume_col].iloc[0][:200]}")
print(f"Sample job desc (first 200 chars): {df[job_col].iloc[0][:200]}")

In [ ]:
# Prepare text pairs and labels
# If dataset only has resume + category (no separate job description),
# we create a synthetic job description from the category

df = df.dropna(subset=[resume_col, label_col])

# Handle case where job_col might be a category rather than full description
if job_col and df[job_col].nunique() < 50:  # It's a category column
    print(f"Job column '{job_col}' appears to be categorical ({df[job_col].nunique()} unique values)")
    # Create job descriptions from categories
    df['job_text'] = df[job_col].apply(lambda x: f"Looking for a candidate with expertise in {x}. "
                                       f"The ideal candidate should have relevant skills and experience in {x}.")
    job_text_col = 'job_text'
else:
    job_text_col = job_col

# Encode labels
le = LabelEncoder()
if df[label_col].dtype == 'object':
    labels = le.fit_transform(df[label_col])
    print(f"Label classes: {dict(zip(le.classes_, le.transform(le.classes_)))}")
else:
    labels = df[label_col].values
    
# Truncate text to max 512 chars for efficiency
resume_texts = df[resume_col].astype(str).apply(lambda x: x[:512]).tolist()
if job_text_col:
    job_texts = df[job_text_col].astype(str).apply(lambda x: x[:512]).tolist()
else:
    job_texts = ['General position requiring relevant skills'] * len(resume_texts)

print(f"\nPrepared {len(resume_texts)} text pairs")
print(f"Label distribution: {np.unique(labels, return_counts=True)}")
print(f"\nSample resume (first 200 chars): {resume_texts[0][:200]}")
print(f"Sample job text (first 200 chars): {job_texts[0][:200]}")

## Step 3: Generate BERT Embeddings
Using Sentence-BERT (all-MiniLM-L6-v2) to generate 384-dimensional embeddings for each resume and job description, then concatenating them into a 768-dimensional feature vector.

In [ ]:
# Load Sentence-BERT model
print("Loading Sentence-BERT model (all-MiniLM-L6-v2)...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f"Model loaded! Embedding dimension: {sbert_model.get_sentence_embedding_dimension()}")

# Generate embeddings
print("\nGenerating resume embeddings...")
resume_embeddings = sbert_model.encode(resume_texts, batch_size=64, show_progress_bar=True,
                                       convert_to_numpy=True)
print(f"Resume embeddings shape: {resume_embeddings.shape}")

print("\nGenerating job description embeddings...")
job_embeddings = sbert_model.encode(job_texts, batch_size=64, show_progress_bar=True,
                                    convert_to_numpy=True)
print(f"Job embeddings shape: {job_embeddings.shape}")

# Concatenate: [resume_embedding | job_embedding] → 768-dim
combined_features = np.concatenate([resume_embeddings, job_embeddings], axis=1)
print(f"\nCombined feature vector shape: {combined_features.shape}")
print(f"Each sample: {combined_features.shape[1]}-dimensional feature vector")

## Step 4: Train-Test Split

In [ ]:
# Split data: 80% train, 10% validation, 10% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    combined_features, labels, test_size=0.1, random_state=42, stratify=labels)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.111, random_state=42, stratify=y_train_val)  # 0.111 of 90% ≈ 10%

print(f"Training set:   {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.LongTensor(y_train).to(device)
X_val_t = torch.FloatTensor(X_val).to(device)
y_val_t = torch.LongTensor(y_val).to(device)
X_test_t = torch.FloatTensor(X_test).to(device)
y_test_t = torch.LongTensor(y_test).to(device)

# Create DataLoaders
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = TensorDataset(X_val_t, y_val_t)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"\nBatch size: 32")
print(f"Training batches: {len(train_loader)}")

## Step 5: Define ANN Architecture
**Architecture:** 768 → 512 → 256 → 128 → 2  
With BatchNorm, Dropout (0.3), and ReLU activations

In [ ]:
class ResumeClassifierANN(nn.Module):
    """
    Artificial Neural Network for Resume Classification
    Input: Concatenated BERT embeddings (resume + job description) = 768-dim
    Output: 2 classes (Accept / Reject)
    """
    def __init__(self, input_dim=768, num_classes=2):
        super(ResumeClassifierANN, self).__init__()
        
        self.network = nn.Sequential(
            # Layer 1: 768 → 512
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Layer 2: 512 → 256
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            # Layer 3: 256 → 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            # Output: 128 → num_classes
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.network(x)

# Determine number of classes
num_classes = len(np.unique(labels))
input_dim = combined_features.shape[1]

model = ResumeClassifierANN(input_dim=input_dim, num_classes=num_classes).to(device)
print(f"\nModel Architecture:")
print(f"Input dimension: {input_dim}")
print(f"Output classes: {num_classes}")
print(f"\n{model}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Step 6: Train the Model

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)

# Training loop
EPOCHS = 50
best_val_acc = 0
patience = 10
patience_counter = 0

history = {
    'train_loss': [], 'val_loss': [],
    'train_acc': [], 'val_acc': []
}

print(f"Training BERT+ANN model for {EPOCHS} epochs...")
print(f"Optimizer: Adam (lr=0.001, weight_decay=1e-4)")
print(f"Scheduler: ReduceLROnPlateau (patience=5)")
print(f"Early stopping patience: {patience}")
print("=" * 70)

for epoch in range(EPOCHS):
    # Training phase
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += batch_y.size(0)
        train_correct += (predicted == batch_y).sum().item()
    
    train_loss /= len(train_loader)
    train_acc = train_correct / train_total
    
    # Validation phase
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            val_total += batch_y.size(0)
            val_correct += (predicted == batch_y).sum().item()
    
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total
    
    # Record history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    
    # Learning rate scheduler
    scheduler.step(val_loss)
    
    # Early stopping check
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_bert_ann_model.pth')
        patience_counter = 0
        marker = ' ✓ (saved)'
    else:
        patience_counter += 1
        marker = ''
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:3d}/{EPOCHS}] | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}%{marker}")
    
    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}! Best val accuracy: {best_val_acc*100:.2f}%")
        break

print(f"\n{'=' * 70}")
print(f"Training complete! Best validation accuracy: {best_val_acc*100:.2f}%")

## Step 7: Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_acc']) + 1)

# Accuracy plot
ax1.plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', markersize=3, label='Training Accuracy', linewidth=2)
ax1.plot(epochs_range, [a*100 for a in history['val_acc']], 'g-s', markersize=3, label='Validation Accuracy', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('BERT + ANN Training Accuracy', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.axhline(y=best_val_acc*100, color='r', linestyle='--', alpha=0.7)
ax1.text(len(history['train_acc'])*0.7, best_val_acc*100+1, f'Best: {best_val_acc*100:.2f}%', 
         color='r', fontweight='bold', fontsize=11)

# Loss plot
ax2.plot(epochs_range, history['train_loss'], 'b-o', markersize=3, label='Training Loss', linewidth=2)
ax2.plot(epochs_range, history['val_loss'], 'r-s', markersize=3, label='Validation Loss', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Loss', fontsize=12)
ax2.set_title('BERT + ANN Training Loss', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('bert_ann_training_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_training_curves.png")

## Step 8: Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_bert_ann_model.pth'))
model.eval()

# Predict on test set
with torch.no_grad():
    test_outputs = model(X_test_t)
    _, y_pred = torch.max(test_outputs, 1)
    
    # Get probabilities for ROC curve
    probs = torch.softmax(test_outputs, dim=1).cpu().numpy()

y_pred_np = y_pred.cpu().numpy()
y_test_np = y_test_t.cpu().numpy()

# Calculate metrics
test_acc = accuracy_score(y_test_np, y_pred_np)
test_prec = precision_score(y_test_np, y_pred_np, average='weighted')
test_rec = recall_score(y_test_np, y_pred_np, average='weighted')
test_f1 = f1_score(y_test_np, y_pred_np, average='weighted')

print("=" * 50)
print("   BERT + ANN TEST SET RESULTS")
print("=" * 50)
print(f"   Accuracy:  {test_acc*100:.2f}%")
print(f"   Precision: {test_prec*100:.2f}%")
print(f"   Recall:    {test_rec*100:.2f}%")
print(f"   F1-Score:  {test_f1*100:.2f}%")
print("=" * 50)

print("\nDetailed Classification Report:")
print(classification_report(y_test_np, y_pred_np))

## Step 9: Confusion Matrix & ROC Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test_np, y_pred_np)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Rejected', 'Accepted'] if num_classes == 2 else range(num_classes),
            yticklabels=['Rejected', 'Accepted'] if num_classes == 2 else range(num_classes))
ax1.set_xlabel('Predicted', fontsize=12, fontweight='bold')
ax1.set_ylabel('Actual', fontsize=12, fontweight='bold')
ax1.set_title(f'Confusion Matrix (Acc: {test_acc*100:.2f}%)', fontsize=13, fontweight='bold')

# ROC Curve (for binary classification)
if num_classes == 2:
    fpr, tpr, _ = roc_curve(y_test_np, probs[:, 1])
    roc_auc = auc(fpr, tpr)
    ax2.plot(fpr, tpr, 'b-', linewidth=2, label=f'BERT+ANN (AUC = {roc_auc:.3f})')
    ax2.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Baseline')
    ax2.set_xlabel('False Positive Rate', fontsize=12)
    ax2.set_ylabel('True Positive Rate', fontsize=12)
    ax2.set_title('ROC Curve', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
else:
    # For multi-class: show per-class accuracy
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    ax2.bar(range(num_classes), per_class_acc * 100, color='#356BB5')
    ax2.set_xlabel('Class', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Per-Class Accuracy', fontsize=13, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('bert_ann_evaluation.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_evaluation.png")

## Step 10: Comparison with Existing Research Papers

In [ ]:
# Comparison with papers from our literature survey
comparison_data = {
    'Study': [
        'Sabarirajan et al. (2025)',
        'Wankhede et al. (2024)',
        'More & Kene (2024)',
        'Jyothsna et al. (2024)',
        'Bevara et al. (2025)',
        'Sriranjani & Kalaiarasi (2025)',
        'Tejaswini et al. (2022)',
        'RizeHire (Ours)'
    ],
    'Technique': [
        'BERT+RoBERTa',
        'NLP+Decision Tree',
        'TF-IDF+ML',
        'Cosine Similarity',
        'SBERT Embeddings',
        'BERT+Random Forest',
        'TF-IDF+ML',
        'SBERT+ANN'
    ],
    'Accuracy/F1': [
        87,  # F1: 0.87
        96,  # Decision Tree accuracy
        90,  # ~88-92%
        87,  # AdaBoost
        78,  # 15.85% nDCG improvement (estimated accuracy)
        87,  # F1-score 87%+
        85,  # text parsing accuracy
        round(test_acc * 100, 2)  # Our actual accuracy
    ],
    'Explainability': [
        'SHAP+LIME',
        'None',
        'None',
        'None',
        'None',
        'Partial',
        'None',
        'SHAP+LIME'
    ],
    'Real-Time': [
        'No', 'No', 'No', 'No', 'No', 'No', 'No', 'Yes (Socket.IO)'
    ]
}

comp_df = pd.DataFrame(comparison_data)

fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#95A5A6'] * 7 + ['#27AE60']
bars = ax.barh(comp_df['Study'], comp_df['Accuracy/F1'], color=colors, edgecolor='white', height=0.6)
ax.set_xlabel('Accuracy / F1-Score (%)', fontsize=12, fontweight='bold')
ax.set_title('Performance Comparison with Latest Research Papers', fontsize=14, fontweight='bold')
ax.set_xlim(0, 105)
ax.grid(True, alpha=0.3, axis='x')

for bar, acc, xai in zip(bars, comp_df['Accuracy/F1'], comp_df['Explainability']):
    label = f'{acc}%'
    if xai in ['SHAP+LIME']:
        label += ' + XAI'
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, label,
            va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('bert_ann_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_comparison.png")
print("\nComparison Table:")
print(comp_df.to_string(index=False))

## Step 11: SHAP Explainability on BERT+ANN Model

In [ ]:
!pip install -q shap

In [ ]:
import shap

# Create a wrapper function for SHAP
def model_predict(X):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.FloatTensor(X).to(device)
        outputs = torch.softmax(model(X_tensor), dim=1)
        return outputs.cpu().numpy()

# Use a small background dataset for SHAP
background = X_train[:100]
explainer = shap.KernelExplainer(model_predict, background)

# Explain a few test samples
test_samples = X_test[:20]
print("Computing SHAP values (this may take a few minutes)...")
shap_values = explainer.shap_values(test_samples)
print("SHAP values computed!")

# Create feature names
feature_names = [f'Resume_BERT_{i}' for i in range(384)] + [f'Job_BERT_{i}' for i in range(384)]

# Summary plot
plt.figure(figsize=(10, 6))
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[1], test_samples, feature_names=feature_names, 
                      max_display=15, show=False)
else:
    shap.summary_plot(shap_values, test_samples, feature_names=feature_names,
                      max_display=15, show=False)
plt.title('SHAP Feature Importance — BERT+ANN Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('bert_ann_shap.png', dpi=200, bbox_inches='tight')
plt.show()
print("Saved: bert_ann_shap.png")

## Step 12: Save Model & Export Results

In [ ]:
# Save final model
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': input_dim,
    'num_classes': num_classes,
    'architecture': '768→512→256→128→2',
    'test_accuracy': test_acc,
    'test_precision': test_prec,
    'test_recall': test_rec,
    'test_f1': test_f1,
    'training_history': history,
    'best_val_accuracy': best_val_acc,
}, 'rizehire_bert_ann_final.pth')

print("Model saved: rizehire_bert_ann_final.pth")

# Print final summary
print("\n" + "=" * 60)
print("       FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"  Model:           BERT (SBERT) + ANN")
print(f"  BERT Model:      all-MiniLM-L6-v2 (384-dim)")
print(f"  ANN Architecture: {input_dim}→512→256→128→{num_classes}")
print(f"  Training Samples: {len(X_train)}")
print(f"  Test Samples:     {len(X_test)}")
print(f"  GPU:             {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  ─────────────────────────────────")
print(f"  Test Accuracy:   {test_acc*100:.2f}%")
print(f"  Test Precision:  {test_prec*100:.2f}%")
print(f"  Test Recall:     {test_rec*100:.2f}%")
print(f"  Test F1-Score:   {test_f1*100:.2f}%")
print(f"  Best Val Acc:    {best_val_acc*100:.2f}%")
print("=" * 60)
print("\n✅ All results saved! Download the following files:")
print("  - rizehire_bert_ann_final.pth (model weights)")
print("  - bert_ann_training_curves.png")
print("  - bert_ann_evaluation.png")
print("  - bert_ann_comparison.png")
print("  - bert_ann_shap.png")

## Step 13: Download All Result Files

In [ ]:
# Download all generated files
from google.colab import files

for f in ['rizehire_bert_ann_final.pth', 
          'bert_ann_training_curves.png',
          'bert_ann_evaluation.png', 
          'bert_ann_comparison.png',
          'bert_ann_shap.png']:
    try:
        files.download(f)
        print(f"Downloaded: {f}")
    except:
        print(f"Download manually: {f}")